Cell 1 — Setup کامل

In [ ]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import networkx as nx

from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv


PROJECT_ROOT = Path(".")

GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"

DAY13_DIR = GRAPH_DIR / "day13_explainability"
DAY16_DIR = GRAPH_DIR / "day16_novel_predictions"
DAY17_DIR = GRAPH_DIR / "day17_novel_explainability"

MODEL_DIR = DAY13_DIR / "saved_models"

DAY17_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GRAPH_DIR:", GRAPH_DIR)
print("DAY16_DIR:", DAY16_DIR)
print("DAY17_DIR:", DAY17_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("DEVICE:", DEVICE)

Cell 2 — Load داده‌های اصلی

In [ ]:
nodes = pd.read_csv(GRAPH_DIR / "graph_nodes.csv")
interaction_edges = pd.read_csv(GRAPH_DIR / "interaction_edges_labeled.csv")
ppi_edges = pd.read_csv(GRAPH_DIR / "ppi_edges.csv")
loc_edges = pd.read_csv(GRAPH_DIR / "colocalization_edges.csv")
node_features = np.load(GRAPH_DIR / "node_features_esm650.npy")

novel_scored = pd.read_csv(
    DAY16_DIR / "novel_candidate_space_hgt_scored.csv"
)

top50 = pd.read_csv(
    DAY16_DIR / "top50_novel_predictions_hgt.csv"
)

print("nodes:", nodes.shape)
print("interaction_edges:", interaction_edges.shape)
print("ppi_edges:", ppi_edges.shape)
print("loc_edges:", loc_edges.shape)
print("node_features:", node_features.shape)
print("novel_scored:", novel_scored.shape)
print("top50:", top50.shape)

display(nodes.head())
display(interaction_edges.head())
display(top50.head(20))

Cell 3 — QC اولیه Day 16 outputs

In [ ]:
qc_day16_path = DAY16_DIR / "day16_novel_prediction_qc.csv"

if qc_day16_path.exists():
    qc_day16 = pd.read_csv(qc_day16_path)
    display(qc_day16)
else:
    print("QC file not found:", qc_day16_path)

print("Novel scored columns:")
print(novel_scored.columns.tolist())

print("Top50 probability range:")
print(top50["prob_hgt_mean"].min(), top50["prob_hgt_mean"].max())

Cell 4 — QC مهم AURKA

این سلول بررسی می‌کند AURKA چرا به‌عنوان E3 وارد دیتاست شده است.

In [ ]:
aurka_node = nodes[
    nodes["gene"].astype(str) == "AURKA"
].copy()

print("AURKA node rows:")
display(aurka_node)

aurka_known_edges = interaction_edges[
    interaction_edges["enz_gene"].astype(str) == "AURKA"
].copy()

print("Known AURKA enzyme-substrate edges:", aurka_known_edges.shape)
display(aurka_known_edges.head(50))

aurka_in_top50 = top50[
    top50["enz_gene"].astype(str) == "AURKA"
].copy()

print("AURKA in top50 novel predictions:", aurka_in_top50.shape)
display(
    aurka_in_top50[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "prob_hgt_std",
            "pair_id"
        ]
    ]
)

Cell 5 — QC AURKA از فایل pair اصلی

In [ ]:
pairs_all = pd.read_csv(
    PROJECT_ROOT / "Data_proc" / "pairs" / "pairs_all_embedding_ready.csv",
    dtype=str,
    low_memory=False
)

aurka_pairs_all = pairs_all[
    pairs_all["enz_gene"].astype(str) == "AURKA"
].copy()

print("AURKA pairs in pairs_all:", aurka_pairs_all.shape)

display(
    aurka_pairs_all[
        [
            "pair_id",
            "enzyme_class",
            "enz_ac",
            "sub_ac",
            "enz_gene",
            "sub_gene",
            "enzyme_type",
            "label",
            "source"
        ]
    ].head(50)
)

print("AURKA unique annotations:")
display(
    aurka_pairs_all[
        [
            "enzyme_class",
            "enz_ac",
            "enz_gene",
            "enzyme_type",
            "source"
        ]
    ]
    .drop_duplicates()
)

Cell 6 — QC همه enzymeهای پرتکرار Top50

این کار مهم است چون AURKA ممکن است تنها مورد مشکوک نباشد.

In [ ]:
top50_enzyme_summary = (
    top50
    .groupby(["enzyme_class", "enz_gene", "enz_ac"])
    .agg(
        n_top50=("pair_id", "count"),
        mean_prob=("prob_hgt_mean", "mean"),
        mean_std=("prob_hgt_std", "mean")
    )
    .reset_index()
    .sort_values("n_top50", ascending=False)
)

display(top50_enzyme_summary)

top50_enzyme_summary.to_csv(
    DAY17_DIR / "top50_enzyme_summary_qc.csv",
    index=False
)

Cell 7 — علامت‌گذاری suspicious enzymeها

فعلاً فقط AURKA را مشکوک می‌گیریم. اگر بعداً از QC مورد دیگری درآمد، اضافه می‌کنیم.

In [ ]:
suspicious_enzymes = {
    "AURKA"
}

top50 = top50.copy()
top50["suspicious_enzyme"] = top50["enz_gene"].isin(suspicious_enzymes)

top50_no_suspicious = (
    top50[
        ~top50["suspicious_enzyme"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Top50 all:", top50.shape)
print("Top50 excluding suspicious:", top50_no_suspicious.shape)

display(
    top50[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "prob_hgt_std",
            "suspicious_enzyme"
        ]
    ].head(50)
)

display(
    top50_no_suspicious[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "prob_hgt_std",
            "pair_id"
        ]
    ].head(30)
)

top50_no_suspicious.to_csv(
    DAY17_DIR / "top50_novel_predictions_no_suspicious.csv",
    index=False
)

Cell 8 — انتخاب کاندیدهای Day17 برای Explainability

اینجا عمداً AURKA را فعلاً کنار می‌گذاریم تا تحلیل اصلی تمیز و قابل دفاع باشد.

In [ ]:
candidate_list = [
    ("USP15", "AKT1"),
    ("USP15", "TP53"),
    ("USP15", "MAPK8"),
    ("USP15", "MAPK3"),
    ("USP15", "TNFAIP3"),
    ("USP10", "CTNNB1"),
    ("USP10", "MAPK3"),
    ("CDC20", "TP53"),
    ("USP20", "CTNNB1"),
    ("USP20", "TRAF6"),
    ("TRIM32", "SQSTM1"),
    ("TRAF4", "AKT1"),
]

selected_rows = []

for enz_gene, sub_gene in candidate_list:
    hit = novel_scored[
        (novel_scored["enz_gene"].astype(str) == enz_gene)
        &
        (novel_scored["sub_gene"].astype(str) == sub_gene)
    ].copy()

    if len(hit) == 0:
        selected_rows.append({
            "enz_gene": enz_gene,
            "sub_gene": sub_gene,
            "found": False,
        })
    else:
        r = hit.sort_values("prob_hgt_mean", ascending=False).iloc[0]

        selected_rows.append({
            "enzyme_class": r["enzyme_class"],
            "enz_gene": r["enz_gene"],
            "sub_gene": r["sub_gene"],
            "enz_ac": r["enz_ac"],
            "sub_ac": r["sub_ac"],
            "enz_node": int(r["enz_node"]),
            "sub_node": int(r["sub_node"]),
            "pair_id": r["pair_id"],
            "prob_hgt_mean": float(r["prob_hgt_mean"]),
            "prob_hgt_std": float(r["prob_hgt_std"]),
            "found": True,
        })

selected_novel = pd.DataFrame(selected_rows)

display(selected_novel)

selected_novel.to_csv(
    DAY17_DIR / "selected_novel_candidates_for_explainability.csv",
    index=False
)

Cell 9 — ساخت HeteroData

In [ ]:
def edge_index_from_df(df):
    return torch.tensor(
        df[["src", "dst"]].values.T,
        dtype=torch.long
    )


hetero_data = HeteroData()

hetero_data["protein"].x = torch.tensor(
    node_features,
    dtype=torch.float32
)

hetero_data["protein", "enzyme_substrate", "protein"].edge_index = edge_index_from_df(
    interaction_edges
)

ppi_edge_index = edge_index_from_df(ppi_edges)
ppi_edge_index = torch.cat(
    [ppi_edge_index, ppi_edge_index[[1, 0], :]],
    dim=1
)

hetero_data["protein", "ppi", "protein"].edge_index = ppi_edge_index

loc_edge_index = edge_index_from_df(loc_edges)
loc_edge_index = torch.cat(
    [loc_edge_index, loc_edge_index[[1, 0], :]],
    dim=1
)

hetero_data["protein", "co_localized", "protein"].edge_index = loc_edge_index

hetero_data = hetero_data.to(DEVICE)

print(hetero_data)

Cell 10 — Load metadata

In [ ]:
ckpt0 = torch.load(
    MODEL_DIR / "hgt_fold0_best.pt",
    map_location="cpu"
)

HGT_METADATA = ckpt0["metadata"]

print("HGT_METADATA:", HGT_METADATA)
print("in_dim:", ckpt0["in_dim"])
print("hidden_dim:", ckpt0["hidden_dim"])
print("emb_dim:", ckpt0["emb_dim"])
print("heads:", ckpt0["heads"])
print("dropout:", ckpt0["dropout"])

Cell 11 — تعریف HGT Model

In [ ]:
class HGTEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim)

        self.hgt1 = HGTConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.hgt2 = HGTConv(
            in_channels=hidden_dim,
            out_channels=out_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = x_dict["protein"]

        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt1(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt2(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}


class HeteroLinkPredictor(nn.Module):
    def __init__(
        self,
        emb_dim=128,
        hidden_dim=128,
        dropout=0.35,
    ):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z_dict, edge_label_index):
        z = z_dict["protein"]

        src = edge_label_index[0]
        dst = edge_label_index[1]

        z_src = z[src]
        z_dst = z[dst]

        h = torch.cat(
            [
                z_src,
                z_dst,
                torch.abs(z_src - z_dst),
                z_src * z_dst,
            ],
            dim=1,
        )

        return self.mlp(h).squeeze(-1)


class HGTLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = HGTEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index):
        edge_index_dict = {
            k: v
            for k, v in data.edge_index_dict.items()
            if k in HGT_METADATA[1]
        }

        z_dict = self.encoder(
            data.x_dict,
            edge_index_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

Cell 12 — Load مدل fold0

In [ ]:
def load_hgt_model(fold=0):
    ckpt = torch.load(
        MODEL_DIR / f"hgt_fold{fold}_best.pt",
        map_location="cpu"
    )

    model = HGTLinkModel(
        in_dim=ckpt["in_dim"],
        hidden_dim=ckpt["hidden_dim"],
        emb_dim=ckpt["emb_dim"],
        heads=ckpt["heads"],
        dropout=ckpt["dropout"],
    ).to(DEVICE)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    return model


model = load_hgt_model(0)

print("HGT fold0 loaded")

Cell 13 — ساخت Neighbor Graph

In [ ]:
neighbor_graph = defaultdict(set)

for df in [
    interaction_edges,
    ppi_edges,
    loc_edges,
]:
    for _, r in df.iterrows():
        s = int(r["src"])
        d = int(r["dst"])

        neighbor_graph[s].add(d)
        neighbor_graph[d].add(s)

node2gene = (
    nodes
    .set_index("node_id")["gene"]
    .fillna("UNKNOWN")
    .to_dict()
)

node2ac = (
    nodes
    .set_index("node_id")["uniprot_ac"]
    .to_dict()
)

print("nodes with neighbors:", len(neighbor_graph))

In [ ]:
def score_edge(
    model,
    data,
    enz_node,
    sub_node,
):
    edge_index = torch.tensor(
        [[int(enz_node)], [int(sub_node)]],
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():
        logit = model(
            data,
            edge_index
        )

        prob = torch.sigmoid(logit).item()

    return prob


def compute_novel_node_importance(
    row,
    top_n_candidates=100,
):
    enz_node = int(row["enz_node"])
    sub_node = int(row["sub_node"])

    base_prob = score_edge(
        model,
        hetero_data,
        enz_node,
        sub_node
    )

    candidate_nodes = list(
        set(neighbor_graph[enz_node])
        |
        set(neighbor_graph[sub_node])
        |
        {enz_node, sub_node}
    )

    candidate_nodes = sorted(
        candidate_nodes,
        key=lambda n: len(neighbor_graph[n]),
        reverse=True
    )[:top_n_candidates]

    rows = []

    for n in candidate_nodes:
        x_backup = hetero_data["protein"].x[n].clone()

        hetero_data["protein"].x[n] = 0.0

        masked_prob = score_edge(
            model,
            hetero_data,
            enz_node,
            sub_node
        )

        hetero_data["protein"].x[n] = x_backup

        rows.append({
            "pair_id": row["pair_id"],
            "enzyme_class": row["enzyme_class"],
            "enz_gene": row["enz_gene"],
            "sub_gene": row["sub_gene"],
            "enz_node": enz_node,
            "sub_node": sub_node,
            "node_id": n,
            "gene": node2gene.get(n, "UNKNOWN"),
            "uniprot": node2ac.get(n, ""),
            "degree": len(neighbor_graph[n]),
            "base_prob_fold0": base_prob,
            "masked_prob": masked_prob,
            "importance_drop": base_prob - masked_prob,
            "prob_hgt_mean_day16": row["prob_hgt_mean"],
            "prob_hgt_std_day16": row["prob_hgt_std"],
        })

    out = (
        pd.DataFrame(rows)
        .sort_values("importance_drop", ascending=False)
        .reset_index(drop=True)
    )

    return out

Cell 14 — تابع scoring و Node Importance

In [ ]:
def score_edge(
    model,
    data,
    enz_node,
    sub_node,
):
    edge_index = torch.tensor(
        [[int(enz_node)], [int(sub_node)]],
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():
        logit = model(
            data,
            edge_index
        )

        prob = torch.sigmoid(logit).item()

    return prob


def compute_novel_node_importance(
    row,
    top_n_candidates=100,
):
    enz_node = int(row["enz_node"])
    sub_node = int(row["sub_node"])

    base_prob = score_edge(
        model,
        hetero_data,
        enz_node,
        sub_node
    )

    candidate_nodes = list(
        set(neighbor_graph[enz_node])
        |
        set(neighbor_graph[sub_node])
        |
        {enz_node, sub_node}
    )

    candidate_nodes = sorted(
        candidate_nodes,
        key=lambda n: len(neighbor_graph[n]),
        reverse=True
    )[:top_n_candidates]

    rows = []

    for n in candidate_nodes:
        x_backup = hetero_data["protein"].x[n].clone()

        hetero_data["protein"].x[n] = 0.0

        masked_prob = score_edge(
            model,
            hetero_data,
            enz_node,
            sub_node
        )

        hetero_data["protein"].x[n] = x_backup

        rows.append({
            "pair_id": row["pair_id"],
            "enzyme_class": row["enzyme_class"],
            "enz_gene": row["enz_gene"],
            "sub_gene": row["sub_gene"],
            "enz_node": enz_node,
            "sub_node": sub_node,
            "node_id": n,
            "gene": node2gene.get(n, "UNKNOWN"),
            "uniprot": node2ac.get(n, ""),
            "degree": len(neighbor_graph[n]),
            "base_prob_fold0": base_prob,
            "masked_prob": masked_prob,
            "importance_drop": base_prob - masked_prob,
            "prob_hgt_mean_day16": row["prob_hgt_mean"],
            "prob_hgt_std_day16": row["prob_hgt_std"],
        })

    out = (
        pd.DataFrame(rows)
        .sort_values("importance_drop", ascending=False)
        .reset_index(drop=True)
    )

    return out

Cell 15 — اجرای Node Importance برای کاندیدهای منتخب

In [ ]:
all_novel_explanations = []

valid_selected = selected_novel[
    selected_novel["found"] == True
].copy()

for _, row in valid_selected.iterrows():
    print("=" * 100)
    print(row["enz_gene"], "->", row["sub_gene"])

    imp = compute_novel_node_importance(
        row,
        top_n_candidates=100
    )

    display(imp.head(15))

    all_novel_explanations.append(imp)

novel_explanations = pd.concat(
    all_novel_explanations,
    ignore_index=True
)

print("novel_explanations:", novel_explanations.shape)

novel_explanations.to_csv(
    DAY17_DIR / "novel_candidate_node_importance.csv",
    index=False
)

Cell 16 — خلاصه Explainability برای مقاله

In [ ]:
novel_explain_summary = []

for pair_id, g in novel_explanations.groupby("pair_id"):
    first = g.iloc[0]

    top_genes = (
        g.sort_values("importance_drop", ascending=False)
        .head(8)["gene"]
        .tolist()
    )

    novel_explain_summary.append({
        "pair_id": pair_id,
        "enzyme_class": first["enzyme_class"],
        "enzyme": first["enz_gene"],
        "substrate": first["sub_gene"],
        "prob_hgt_mean": first["prob_hgt_mean_day16"],
        "prob_hgt_std": first["prob_hgt_std_day16"],
        "base_prob_fold0": first["base_prob_fold0"],
        "top_explanatory_genes": "; ".join(top_genes),
    })

novel_explain_summary = (
    pd.DataFrame(novel_explain_summary)
    .sort_values("prob_hgt_mean", ascending=False)
)

display(novel_explain_summary)

novel_explain_summary.to_csv(
    DAY17_DIR / "novel_candidate_explainability_summary.csv",
    index=False
)

Cell 17 — شکل شبکه برای یک Novel Candidate

In [ ]:
def plot_novel_explanation_network(
    pair_id,
    top_n=15,
):
    tmp = (
        novel_explanations[
            novel_explanations["pair_id"] == pair_id
        ]
        .sort_values("importance_drop", ascending=False)
        .head(top_n)
        .copy()
    )

    enz = tmp["enz_gene"].iloc[0]
    sub = tmp["sub_gene"].iloc[0]

    G = nx.Graph()

    if (tmp["gene"] == enz).any():
        enz_imp = float(tmp.loc[tmp["gene"] == enz, "importance_drop"].max())
    else:
        enz_imp = float(tmp["importance_drop"].max())

    if (tmp["gene"] == sub).any():
        sub_imp = float(tmp.loc[tmp["gene"] == sub, "importance_drop"].max())
    else:
        sub_imp = float(tmp["importance_drop"].max())

    G.add_node(enz, importance=enz_imp)
    G.add_node(sub, importance=sub_imp)
    G.add_edge(enz, sub, weight=1.0)

    for _, r in tmp.iterrows():
        gene = r["gene"]
        imp = float(r["importance_drop"])

        if gene in [enz, sub]:
            continue

        G.add_node(gene, importance=imp)
        G.add_edge(enz, gene, weight=imp)
        G.add_edge(sub, gene, weight=imp)

    pos = {
        enz: (-0.35, 0),
        sub: (0.35, 0),
    }

    neighbors = [
        n for n in G.nodes()
        if n not in [enz, sub]
    ]

    angles = np.linspace(
        0,
        2*np.pi,
        len(neighbors),
        endpoint=False
    )

    radius = 2.8

    for n, a in zip(neighbors, angles):
        pos[n] = (
            radius*np.cos(a),
            radius*np.sin(a)
        )

    importances = np.array([
        G.nodes[n].get("importance", 0.0)
        for n in G.nodes()
    ])

    imp_min = importances.min()
    imp_max = importances.max()

    def scale_size(
        x,
        min_size=900,
        max_size=4200,
    ):
        if imp_max == imp_min:
            return 1500

        return min_size + (
            (x - imp_min)
            /
            (imp_max - imp_min)
        ) * (max_size - min_size)

    node_sizes = [
        scale_size(G.nodes[n].get("importance", 0.0))
        for n in G.nodes()
    ]

    node_sizes = [
        max(s, 3900) if n in [enz, sub] else s
        for n, s in zip(G.nodes(), node_sizes)
    ]

    plt.figure(figsize=(14, 12))

    nx.draw_networkx_edges(
        G,
        pos,
        alpha=0.35,
        width=1.5
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        node_size=node_sizes,
        linewidths=1.8,
        edgecolors="black"
    )

    nx.draw_networkx_labels(
        G,
        pos,
        font_size=10,
        font_weight="bold"
    )

    plt.title(
        f"Novel Candidate Explanation Network: {enz} → {sub}",
        fontsize=18
    )

    plt.axis("off")
    plt.tight_layout()

    safe = f"{enz}_to_{sub}".replace("/", "_")

    plt.savefig(
        DAY17_DIR / f"figure_novel_explanation_network_{safe}.png",
        dpi=400,
        bbox_inches="tight"
    )

    plt.savefig(
        DAY17_DIR / f"figure_novel_explanation_network_{safe}.pdf",
        bbox_inches="tight"
    )

    plt.show()

Cell 18 — رسم شکل برای بهترین کاندید

In [ ]:
CASE_PAIR = novel_explain_summary.iloc[0]["pair_id"]

print("CASE_PAIR:", CASE_PAIR)

display(
    novel_explanations[
        novel_explanations["pair_id"] == CASE_PAIR
    ]
    .sort_values("importance_drop", ascending=False)
    .head(15)
)

plot_novel_explanation_network(
    CASE_PAIR,
    top_n=15
)

Cell 19 — Literature Evidence Scaffold

In [ ]:
literature_scaffold = novel_explain_summary.copy()

literature_scaffold["known_direct_interaction_in_training"] = False
literature_scaffold["literature_status"] = "to_check"
literature_scaffold["biological_context"] = ""
literature_scaffold["pmid_or_source"] = ""
literature_scaffold["final_priority"] = ""

display(literature_scaffold)

literature_scaffold.to_csv(
    DAY17_DIR / "novel_candidate_literature_scaffold.csv",
    index=False
)

بریم ادامه Day 17 با Relation Dependency. این بخش نشان می‌دهد هر Novel Candidate بیشتر به کدام نوع رابطه وابسته است:

* PPI
* Co-localization
* Enzyme-substrate context

Cell 20 — ساخت نسخه‌های مختلف گراف برای Ablation

In [ ]:
def make_hetero_data_variant(
    use_enzyme_substrate=True,
    use_ppi=True,
    use_colocalization=True,
):
    data = HeteroData()

    data["protein"].x = torch.tensor(
        node_features,
        dtype=torch.float32
    )

    if use_enzyme_substrate:
        data["protein", "enzyme_substrate", "protein"].edge_index = edge_index_from_df(
            interaction_edges
        )
    else:
        data["protein", "enzyme_substrate", "protein"].edge_index = torch.empty(
            (2, 0),
            dtype=torch.long
        )

    if use_ppi:
        ppi_edge_index = edge_index_from_df(ppi_edges)
        ppi_edge_index = torch.cat(
            [ppi_edge_index, ppi_edge_index[[1, 0], :]],
            dim=1
        )
        data["protein", "ppi", "protein"].edge_index = ppi_edge_index
    else:
        data["protein", "ppi", "protein"].edge_index = torch.empty(
            (2, 0),
            dtype=torch.long
        )

    if use_colocalization:
        loc_edge_index = edge_index_from_df(loc_edges)
        loc_edge_index = torch.cat(
            [loc_edge_index, loc_edge_index[[1, 0], :]],
            dim=1
        )
        data["protein", "co_localized", "protein"].edge_index = loc_edge_index
    else:
        data["protein", "co_localized", "protein"].edge_index = torch.empty(
            (2, 0),
            dtype=torch.long
        )

    return data.to(DEVICE)

Cell 21 — تعریف سناریوهای Relation Ablation

In [ ]:
relation_scenarios = {
    "full": {
        "use_enzyme_substrate": True,
        "use_ppi": True,
        "use_colocalization": True,
    },
    "no_ppi": {
        "use_enzyme_substrate": True,
        "use_ppi": False,
        "use_colocalization": True,
    },
    "no_colocalization": {
        "use_enzyme_substrate": True,
        "use_ppi": True,
        "use_colocalization": False,
    },
    "no_enzyme_substrate_context": {
        "use_enzyme_substrate": False,
        "use_ppi": True,
        "use_colocalization": True,
    },
    "only_ppi": {
        "use_enzyme_substrate": False,
        "use_ppi": True,
        "use_colocalization": False,
    },
    "only_colocalization": {
        "use_enzyme_substrate": False,
        "use_ppi": False,
        "use_colocalization": True,
    },
    "only_enzyme_substrate_context": {
        "use_enzyme_substrate": True,
        "use_ppi": False,
        "use_colocalization": False,
    },
}

Cell 22 — ساخت همه Graph Variantها

In [ ]:
hetero_variants = {}

for name, cfg in relation_scenarios.items():
    print("building:", name)

    hetero_variants[name] = make_hetero_data_variant(
        **cfg
    )

print("done")

Cell 23 — محاسبه Relation Dependency برای candidateهای منتخب

In [ ]:
relation_rows = []

valid_selected = selected_novel[
    selected_novel["found"] == True
].copy()

for _, row in valid_selected.iterrows():
    enz_node = int(row["enz_node"])
    sub_node = int(row["sub_node"])

    probs = {}

    print("=" * 100)
    print(row["enz_gene"], "->", row["sub_gene"])

    for scenario_name, data_variant in hetero_variants.items():
        p = score_edge(
            model,
            data_variant,
            enz_node,
            sub_node
        )

        probs[scenario_name] = p

        print(scenario_name, p)

    full_prob = probs["full"]

    relation_rows.append({
        "pair_id": row["pair_id"],
        "enzyme_class": row["enzyme_class"],
        "enzyme": row["enz_gene"],
        "substrate": row["sub_gene"],
        "prob_hgt_mean_day16": row["prob_hgt_mean"],
        "prob_hgt_std_day16": row["prob_hgt_std"],
        "prob_full_fold0": probs["full"],
        "prob_no_ppi": probs["no_ppi"],
        "prob_no_colocalization": probs["no_colocalization"],
        "prob_no_enzyme_substrate_context": probs["no_enzyme_substrate_context"],
        "prob_only_ppi": probs["only_ppi"],
        "prob_only_colocalization": probs["only_colocalization"],
        "prob_only_enzyme_substrate_context": probs["only_enzyme_substrate_context"],
        "drop_no_ppi": full_prob - probs["no_ppi"],
        "drop_no_colocalization": full_prob - probs["no_colocalization"],
        "drop_no_enzyme_substrate_context": full_prob - probs["no_enzyme_substrate_context"],
    })

relation_dependency = pd.DataFrame(relation_rows)

display(relation_dependency)

relation_dependency.to_csv(
    DAY17_DIR / "novel_candidate_relation_dependency.csv",
    index=False
)

Cell 24 — محاسبه Relative Importance نوع رابطه‌ها

In [ ]:
rel_imp = relation_dependency.copy()

drop_cols = [
    "drop_no_ppi",
    "drop_no_colocalization",
    "drop_no_enzyme_substrate_context",
]

for c in drop_cols:
    rel_imp[c] = rel_imp[c].clip(lower=0)

rel_imp["total_drop"] = rel_imp[drop_cols].sum(axis=1)

rel_imp["ppi_importance_pct"] = (
    rel_imp["drop_no_ppi"] / rel_imp["total_drop"] * 100
)

rel_imp["colocalization_importance_pct"] = (
    rel_imp["drop_no_colocalization"] / rel_imp["total_drop"] * 100
)

rel_imp["enzyme_substrate_context_importance_pct"] = (
    rel_imp["drop_no_enzyme_substrate_context"] / rel_imp["total_drop"] * 100
)

display(
    rel_imp[
        [
            "enzyme",
            "substrate",
            "ppi_importance_pct",
            "colocalization_importance_pct",
            "enzyme_substrate_context_importance_pct",
            "total_drop"
        ]
    ]
)

rel_imp.to_csv(
    DAY17_DIR / "novel_candidate_relation_importance_percent.csv",
    index=False
)

Cell 25 — Figure: Relation Dependency Heatmap

In [ ]:
heat_df = rel_imp[
    [
        "enzyme",
        "substrate",
        "ppi_importance_pct",
        "colocalization_importance_pct",
        "enzyme_substrate_context_importance_pct",
    ]
].copy()

heat_df["candidate"] = (
    heat_df["enzyme"] + " → " + heat_df["substrate"]
)

heat_mat = heat_df.set_index("candidate")[
    [
        "ppi_importance_pct",
        "colocalization_importance_pct",
        "enzyme_substrate_context_importance_pct",
    ]
]

plt.figure(figsize=(9, 6))

plt.imshow(
    heat_mat.values,
    aspect="auto"
)

plt.xticks(
    ticks=np.arange(heat_mat.shape[1]),
    labels=[
        "PPI",
        "Co-localization",
        "Enzyme-substrate context"
    ],
    rotation=30,
    ha="right"
)

plt.yticks(
    ticks=np.arange(heat_mat.shape[0]),
    labels=heat_mat.index
)

plt.colorbar(
    label="Relative importance (%)"
)

plt.title(
    "Relation Dependency of Novel Candidate Predictions"
)

plt.tight_layout()

plt.savefig(
    DAY17_DIR / "figure_novel_relation_dependency_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Cell 26 — Figure: Stacked Bar Plot

In [ ]:
plot_df = heat_mat.copy()

plt.figure(figsize=(10, 6))

bottom = np.zeros(len(plot_df))

for col in plot_df.columns:
    values = plot_df[col].values

    plt.barh(
        plot_df.index,
        values,
        left=bottom,
        label=col.replace("_importance_pct", "")
    )

    bottom += values

plt.xlabel("Relative importance (%)")
plt.ylabel("Novel candidate")

plt.title(
    "Relative Contribution of Graph Relation Types"
)

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    DAY17_DIR / "figure_novel_relation_dependency_stacked_bar.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Cell 27 — ساخت جدول نهایی Day17 Candidate Priority

In [ ]:
priority_table = novel_explain_summary.merge(
    rel_imp[
        [
            "pair_id",
            "ppi_importance_pct",
            "colocalization_importance_pct",
            "enzyme_substrate_context_importance_pct",
            "total_drop"
        ]
    ],
    on="pair_id",
    how="left"
)

priority_table["max_relation_dependency"] = priority_table[
    [
        "ppi_importance_pct",
        "colocalization_importance_pct",
        "enzyme_substrate_context_importance_pct"
    ]
].max(axis=1)

priority_table["priority_score"] = (
    priority_table["prob_hgt_mean"]
    *
    (1 / (1 + priority_table["prob_hgt_std"]))
    *
    (1 + priority_table["total_drop"])
)

priority_table = priority_table.sort_values(
    "priority_score",
    ascending=False
)

display(priority_table)

priority_table.to_csv(
    DAY17_DIR / "day17_final_novel_candidate_priority_table.csv",
    index=False
)